In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.5 Preference optimization

> 🎯 **Goal:** Move beyond a single gold answer and teach the model from pairs of
> 'this reply is better than that one'.

**Why this matters.** SFT (l7.2) teaches the model *a* correct answer. But for open
ended questions there is rarely one right reply, there are better and worse ones:
more helpful, more honest, less rude. Preference optimization is the alignment step
that comes after SFT and tunes the model toward what humans actually prefer. It is
the stage that gives modern assistants their polish.

**The intuition.** Instead of a single graded essay, imagine handing the student two
answers to the same question and saying 'this one is better'. They do not need a
perfect model answer, just a steady stream of comparisons, and from many such
'A beats B' judgments they learn the *taste* you are trying to instill. A
**preference** is exactly that comparison, and a **reward** is the score that
captures how good a single answer is.

**The mechanics.** The classic recipe (RLHF) trains a separate reward model and then
runs reinforcement learning, which is fiddly. **DPO** (Direct Preference
Optimization) skips both. It works directly on `(chosen, rejected)` pairs and a
**frozen reference model** (usually the SFT model itself). The DPO loss is low when
the policy raises the chosen reply's likelihood *relative to* the rejected one, by
more than the reference did, and high when it gets the preference backwards. No
reward model, no RL loop, just a clean loss over preference pairs. Below we call
`dpo_loss` twice to confirm it punishes the wrong ordering and rewards the right
one.

In [ ]:
from pocketlm import dpo_loss

ref_chosen, ref_rejected = torch.tensor(-2.0), torch.tensor(-2.0)
good = dpo_loss(torch.tensor(-1.0), torch.tensor(-3.0), ref_chosen, ref_rejected)
bad = dpo_loss(torch.tensor(-3.0), torch.tensor(-1.0), ref_chosen, ref_rejected)
print("loss when policy prefers the chosen reply: ", round(good.item(), 3))
print("loss when policy prefers the rejected reply:", round(bad.item(), 3))

**What just happened.** We fed `dpo_loss` two scenarios with the same reference.
When the policy assigned higher likelihood to the chosen reply than the rejected
one, the loss came out low. When it preferred the rejected reply, the loss came out
high. That asymmetry is the entire training signal: minimizing this loss over many
pairs pushes the model to rank human-preferred answers above the rest, which is the
alignment polish that sits on top of SFT.

⚠️ **Common pitfalls**
- Dropping the reference model. DPO measures improvement *relative to* the frozen
  reference. Without it the model can drift far from the SFT behavior and degrade.
- Noisy or contradictory preference labels. If 'chosen' and 'rejected' are
  inconsistent across your data, the signal cancels out and nothing useful is
  learned.
- Confusing reward with preference. A reward scores one answer, a preference compares
  two. DPO learns straight from comparisons and never needs an explicit reward model.

🏋️ **Try it yourself.** Make the chosen and rejected log-probabilities nearly equal
(say -2.0 and -2.1) and rerun. Watch the loss sit near its midpoint, the model is
almost indifferent. Then widen the gap and see the loss drop, the clearer the
preference, the stronger the push.

In [ ]:
assert good < bad